In [41]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random
import pickle
import math

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

# Hyperparameters
batch_size = 64
block_size = 128
max_iters = 6000
learning_rate = 1e-4
eval_iters = 500        
n_embd = 384
n_head = 4
n_layer = 6
dropout = 0.3

print(device)

cuda


In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import os

# List your Drive root to find the exact folder/file name
print(os.listdir('/content/drive/MyDrive'))

['AP ASSIGNMENT-3 (1).pdf', 'AP ASSIGNMENT-3.pdf', 'Colab Notebooks', 'Anirudh.gsheet', 'House_Rent_Dataset (1).csv', 'Untitled spreadsheet (1).gsheet', 'Untitled spreadsheet.gsheet', 'Student_Marks_and_Grades (3).gsheet', 'Student_Marks_and_Grades (2).gsheet', 'Student_Marks_and_Grades (1).gsheet', 'Student_Marks_and_Grades.gsheet', 'House_Rent_Dataset (1).gsheet', 'House_Rent_Dataset (2).xlsx', 'Edit it.gdoc', 'UST24R0101004 (5).pdf', 'UST24R0101004.pdf', 'Assignment-1 java.pdf', 'UST24R0101004 (1).pdf', 'UST24R0101004 (2).pdf', 'diabetes.csv', 'UST24R0101004 (3).pdf', 'Google AI Studio', 'UST24R0101004 (4).pdf', 'UST24R0101004.ipynb - Colab.pdf', 'Screenshot_20260114_143351_Chrome.jpg', '12th sharda Maths.pdf', '12th sharda Maths.gdoc', 'Delinquency_prediction_dataset (2).csv', 'playground-series-s6e2.zip', 'train[1].csv', 'train[1].gsheet', 'sample_submission.csv', 'test.csv', 'train.csv', 'train.gsheet', 'archive', 'archive (1).zip', 'DOC-20260413-WA0003..pdf', 'DAA Notes.pdf', 'b

In [44]:
# Load the training text and create vocabulary
with open('/content/drive/MyDrive/full_book.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)
print(f"Dataset size: {len(text):,} characters")
print(f"Unique characters: {vocab_size}")

Dataset size: 886,103 characters
Unique characters: 123


In [45]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [46]:
# Load data directly into memory
with open('/content/drive/MyDrive/full_book.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Removed the 100k character limit to use the full book text
print(f"Dataset size: {len(text):,} characters")

# Encode entire dataset
data = torch.tensor(encode(text), dtype=torch.long)

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

Dataset size: 886,103 characters
Train: 797,492 tokens | Val: 88,611 tokens


In [47]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with fused QKV projection"""
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        
        # Fused QKV projection (more efficient than 3 separate projections)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        # Causal mask
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))
                             .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        
        # Fused QKV computation
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        
        # Reshape for multi-head attention: (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # Use PyTorch's scaled_dot_product_attention (uses Flash Attention when available)
        # Falls back to efficient implementation on MPS/CPU
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0)
        
        # Reshape back: (B, n_head, T, head_dim) -> (B, T, C)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.c_proj(out)
        out = self.dropout(out)
        return out

In [48]:
class FeedForward(nn.Module):
    """FFN with GELU activation (current standard)"""
    
    def __init__(self, n_embd):
        super().__init__()
        # 4x expansion is standard, GELU is the modern default
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [49]:
class Block(nn.Module):
    """Transformer block with Pre-LayerNorm (more stable training)"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        # Pre-LN: normalize BEFORE attention/FFN (GPT-2 style, more stable)
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [50]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # Weight tying: share weights between embedding and output projection
        self.token_embedding_table.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
        # Scale residual projections (GPT-2 style initialization)
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        """Generation with temperature and top-k sampling"""
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling (better quality than pure sampling)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size)
m = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in m.parameters()):,}")

Model parameters: 10,734,720


In [51]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [52]:
# Optimizer with cosine LR schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr(it):
    warmup_iters = 100  # slightly longer warmup for more data
    min_lr = learning_rate / 10
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# Track best model for checkpointing
best_val_loss = float('inf')

for iter in range(max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    if iter % 500 == 0:
        losses = estimate_loss()
        print(f"step {iter:4d} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")
        
        # Save the model if validation loss improves (Best Model Checkpointing)
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            with open('best_model.pkl', 'wb') as f:
                pickle.dump(model, f)
            print(f"New Best Point Reached of Model and saved!!!  Val loss: {best_val_loss:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"\nFinal training loss: {loss.item():.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")

step    0 | train: 4.8528 | val: 4.8524
New Best Point Reached of Model and saved!!!  Val loss: 4.8524
step  500 | train: 2.2008 | val: 2.3686
New Best Point Reached of Model and saved!!!  Val loss: 2.3686
step 1000 | train: 1.6554 | val: 1.9987
New Best Point Reached of Model and saved!!!  Val loss: 1.9987
step 1500 | train: 1.4494 | val: 1.8273
New Best Point Reached of Model and saved!!!  Val loss: 1.8273
step 2000 | train: 1.3266 | val: 1.7355
New Best Point Reached of Model and saved!!!  Val loss: 1.7355
step 2500 | train: 1.2552 | val: 1.7014
New Best Point Reached of Model and saved!!!  Val loss: 1.7014
step 3000 | train: 1.1915 | val: 1.6444
New Best Point Reached of Model and saved!!!  Val loss: 1.6444
step 3500 | train: 1.1548 | val: 1.6178
New Best Point Reached of Model and saved!!!  Val loss: 1.6178
step 4000 | train: 1.1238 | val: 1.5977
New Best Point Reached of Model and saved!!!  Val loss: 1.5977
step 4500 | train: 1.1057 | val: 1.5903
New Best Point Reached of Model a

In [68]:
path = '/content/drive/MyDrive/LLM_Training/model-final.pth'
def save_model(model, path):
    # We save ONLY the weights (state_dict), not the class structure
    torch.save(model.state_dict(), path)
    print(f"Weights saved to {path}")

In [69]:
def load_model(path, vocab_size):
    # Step A: Initialize a FRESH instance of the class
    model = GPTLanguageModel(vocab_size)
    
    # Step B: Load the weights dictionary from the file
    state_dict = torch.load(path)

    # Step C: Copy the weights into the model instance
    model.load_state_dict(state_dict)

    # Step D: Set to evaluation mode
    model.eval()

    print("Model weights loaded successfully!")
    return model

In [70]:
save_model(model, path)

Weights saved to /content/drive/MyDrive/LLM_Training/model-final.pth


In [71]:
model = load_model(path, vocab_size)
model.to(device)

Model weights loaded successfully!


GPTLanguageModel(
  (token_embedding_table): Embedding(123, 384)
  (position_embedding_table): Embedding(128, 384)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=384, out_features=1152, bias=False)
        (c_proj): Linear(in_features=384, out_features=384, bias=False)
        (dropout): Dropout(p=0.3, inplace=False)
      )
      (ln2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.3, inplace=False)
        )
      )
    )
    (1): Block(
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=384, out_features=

In [75]:
prompt = "Neural Networks and Deep Learning?"
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context.unsqueeze(0), max_new_tokens=1000, temperature=0.8, top_k=40)[0].tolist())
print("\n---Generated Text---\n")
print(generated_chars)


---Generated Text---

Neural Networks and Deep Learning? Every As potion Hey and Ramous Psitable, and prevantage algorithms decision by accodences on. Such application the position of why discaling represent the test pexeriments
rause above an is human ince moderned in this traivieve the graphiles, and weighthed
will-configurately computes that size
a actual design above a train of $1,10$ and that the step chige how functure.
False which $0$ to from the
developed othetures we are guper firsting out in that wo reason eversality is and very substation.
The idea using the number of the biases making at good to overfit their term still.  And so out but an poinsiblity to the learning
my undaments of a perhaps becomer such as call in trainstain data.  The sigmoid function (e.e., one the of the resulting a neurons.  In this is into
intuiting the correctly chosith to we the use activation of the training data.  The way to is a function of the
fully-connection clear, by many complexity of the 